# Corrección de las respuestas equivocadas a las preguntas del exámen 1 de la clase de laboratorio de aprendizaje estadístico
|                |   |
:----------------|---|
| **Nombre**     |  Juan Pedro Ley Valdez |
| **Fecha**      |  04 de marzo del 2026 |
| **Expediente** | 746385  |

**Entrega un modelo para predecir las ventas de una tienda dados los factores restantes.**

* Utiliza sklearn para el modelo.

* Utiliza escalamiento para los factores numéricos, y transformaciones para los factores cualitativos.

* Calcula los p-values de los factores utilizando los resultados de sklearn (NO STATSMODELS). Interpreta tus resultados.

In [8]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [3]:
df = pd.read_excel('/content/stores_lae_exam1.xlsx')
df.head()

,Sales,Advertising,Store_Size,Competitors,Years_Active,Manager_Experience,Location,Season,Store_Type,Customer_Satisfaction_Score,Inventory_Turnover_Rate,Employee_Turnover_Rate
0,334.45,17.98,3788.12,2.0,20.0,NaN,Los Angeles,NaN,Outlet,NaN,3.43,12.10
1,337.93,14.17,3773.29,NaN,NaN,14.0,New York,Spring,Outlet,68.10,3.33,16.05
2,281.62,18.89,NaN,5.0,NaN,6.0,Houston,Spring,NaN,83.81,NaN,8.13
3,467.81,24.14,3834.95,2.0,20.0,14.0,New York,Summer,NaN,76.21,4.42,4.59
4,376.10,13.60,4366.41,6.0,8.0,19.0,Chicago,Summer,Outlet,76.69,9.01,8.61


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Sales                        4471 non-null   object 
 1   Advertising                  4524 non-null   object 
 2   Store_Size                   4485 non-null   object 
 3   Competitors                  4499 non-null   float64
 4   Years_Active                 4484 non-null   float64
 5   Manager_Experience           4531 non-null   float64
 6   Location                     4491 non-null   object 
 7   Season                       4470 non-null   object 
 8   Store_Type                   4497 non-null   object 
 9   Customer_Satisfaction_Score  4481 non-null   object 
 10  Inventory_Turnover_Rate      4506 non-null   object 
 11  Employee_Turnover_Rate       4525 non-null   object 
dtypes: float64(3), object(9)
memory usage: 468.9+ KB


In [9]:
# Limpieza
columnas_a_numerico = [
    'Sales',
    'Advertising',
    'Customer_Satisfaction_Score',
    'Inventory_Turnover_Rate',
    'Employee_Turnover_Rate',
    'Store_Size'
]
for col in columnas_a_numerico:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace(r'[$,% ]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')

df_limpio = df.dropna().reset_index(drop=True)
print(df_limpio.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1385 entries, 0 to 1384
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Sales                        1385 non-null   float64
 1   Advertising                  1385 non-null   float64
 2   Store_Size                   1385 non-null   float64
 3   Competitors                  1385 non-null   float64
 4   Years_Active                 1385 non-null   float64
 5   Manager_Experience           1385 non-null   float64
 6   Location                     1385 non-null   object 
 7   Season                       1385 non-null   object 
 8   Store_Type                   1385 non-null   object 
 9   Customer_Satisfaction_Score  1385 non-null   float64
 10  Inventory_Turnover_Rate      1385 non-null   float64
 11  Employee_Turnover_Rate       1385 non-null   float64
dtypes: float64(9), object(3)
memory usage: 130.0+ KB
None


## Escalamiento y transformaciones

In [11]:

X = df_limpio.drop('Sales', axis=1)
y = df_limpio['Sales']

numerical_features = ['Competitors', 'Years_Active', 'Manager_Experience', 'Store_Size']
categorical_features = ['Location', 'Season', 'Store_Type']

# Crear el preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])

# Crear el pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Entrenar el modelo
pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Competitors',
                                                   'Years_Active',
                                                   'Manager_Experience',
                                                   'Store_Size']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['Location', 'Season',
                                                   'Store_Type'])])),
                ('model', LinearRegression())])

## Cálculo de p-values

In [17]:
# Obtener la matriz X transformada (escalada y codificada)
X_trans = pipeline.named_steps['preprocessor'].transform(X)

# Añadir la columna del intercepto (columna de 1s al inicio)
X_mat = np.c_[np.ones((X_trans.shape[0], 1)), X_trans]

# Predicciones del modelo
y_pred = pipeline.predict(X)

# Extracción de los coeficientes del modelo (Intercepto seguido de las betas)
modelo = pipeline.named_steps['model']
betas = np.concatenate([[modelo.intercept_], modelo.coef_])


# 1. Calcular RSS (Residual Sum of Squares)
rss = np.sum((y - y_pred)**2)

# 2. Calcular RSE (Residual Standard Error)
n = X_mat.shape[0]  # número de observaciones
p = X_mat.shape[1]  # número total de parámetros (incluyendo el intercepto)
grados_libertad = n - p
rse_sq = rss / grados_libertad # Esto es rse**2 directamente

# 3. Calcular var_beta
var_beta = np.linalg.inv(X_mat.T @ X_mat) * rse_sq

# 4. Calcular std_beta (Error Estándar de cada coeficiente)
std_beta = np.sqrt(var_beta.diagonal())

# 5. Calcular estadístico t
t_stat = betas / std_beta

# 6. Calcular p-value utilizando la distribución t de Student a dos colas
p_values = [2 * (1 - stats.t.cdf(np.abs(t), df=grados_libertad)) for t in t_stat]

# --- PRESENTACIÓN DE RESULTADOS ---
# Obtener los nombres de las columnas transformadas
cat_encoder = pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_names = cat_encoder.get_feature_names_out(categorical_features)
nombres_parametros = ['Intercepto'] + numerical_features + list(cat_names)

# Crear un DataFrame para visualizar los resultados claramente
resultados = pd.DataFrame({
    'Factor': nombres_parametros,
    'Coeficiente (Beta)': betas,
    'Std_Beta': std_beta,
    'Estadístico t': t_stat,
    'P-Value': p_values
})

resultados['Significativo (alpha=0.05)'] = np.where(resultados['P-Value'] < 0.05, 'Sí', 'No')

resultados['P-Value'] = resultados['P-Value'].map('{:.5f}'.format)



print(resultados)

                  Factor  Coeficiente (Beta)  Std_Beta  Estadístico t  \
0             Intercepto          350.753946  3.328765     105.370606   
1            Competitors           -1.152327  1.100692      -1.046912   
2           Years_Active            5.460979  1.100972       4.960142   
3     Manager_Experience            4.980627  1.099442       4.530142   
4             Store_Size           28.177150  1.103597      25.532096   
5       Location_Houston           -6.253705  3.367982      -1.856810   
6   Location_Los Angeles            3.315136  3.487876       0.950474   
7         Location_Miami           -4.210989  3.386519      -1.243456   
8      Location_New York           22.723921  3.470087       6.548516   
9          Season_Spring          -15.849436  3.096465      -5.118558   
10         Season_Summer           17.720444  3.119683       5.680206   
11         Season_Winter          -20.633589  3.088395      -6.681008   
12  Store_Type_Franchise           -6.670027  2.675

## Interpretación de los resultados


Para leer esta tabla de forma sencilla, la regla principal es mirar la columna del p-Value. Si el valor es menor a 0.05, el factor tiene un impacto real y comprobado en las ventas. Si es mayor, significa que ese factor no hace diferencia en este modelo.

**Factores que SÍ impactan las ventas:**
* **Impulsan las ventas hacia arriba:** Tener un local más grande (`Store_Size`), llevar más años operando (`Years_Active`), tener un gerente con más experiencia (`Manager_Experience`), la temporada de verano (`Summer`) y estar ubicado en Nueva York (`New York`).
* **Empujan las ventas hacia abajo:** Las temporadas de primavera (`Spring`) e invierno (`Winter`), así como ser una tienda tipo franquicia (`Franchise`) o un outlet (`Outlet`).

**Factores que NO impactan las ventas:**
* **Competidores (`Competitors`):** La cantidad de competencia cercana no afecta la cantidad de ventas de la tienda.
* **Otras ciudades:** Estar en Houston, Los Ángeles o Miami no genera una diferencia real en las ventas.

## Preguntas de correcciones
**¿Cuál fue el error?**
No recordar el procedimiento algebraico y matricial para calcular manualmente los *p-values* asociados a los coeficientes ($\beta$) del modelo de regresión lineal.

**¿Cuál es la corrección?**
Implementar las fórmulas matemáticas proporcionadas en el Laboratorio 3. Esto incluye calcular la suma de cuadrados de los residuos (RSS), el error estándar residual (RSE), la matriz de varianza de los coeficientes y el estadístico *t* para derivar los *p-values* sin depender de librerías externas.

**¿Por qué se cometió el error?**
Por una dificultad para rastrear las referencias del curso. Al no tener presente en qué práctica de laboratorio exacta se había desarrollado este procedimiento manual, fue imposible ubicar el material de consulta durante la evaluación.

**¿Cómo se puede evitar este error en el futuro?**
Creando un cheat sheet para mantener una hoja de cálculo o documento simple que mapee conceptos clave con su ubicación del documento en el que se encuentra.






